# #4 Cleanup and markers

## Purpose

Cleanup dataset by dropping doublets identified during annotation and removing resulting unifurcations from the trees. Then, identify marker genes for each cell subtype.

## Setup

In [1]:
from pathlib import Path
import treedata as td
import pandas as pd
import scanpy as sc
import seaborn as sns

from devmap.config import set_theme, get_paths
from devmap.topology import compress_single_child_internal_nodes

set_theme()
base_path, plots_path, results_path = get_paths()
data_path = base_path / "data"

## Load data

In [ ]:
tdata = td.read_h5td("/lab/wcolgan_scratch/combined_counts.h5td")
obs = pd.read_csv("/lab/wcolgan_scratch/combined_obs.csv", index_col=0)
cell_types = pd.read_csv("/lab/wcolgan_scratch/cell_types.csv", index_col=0)
obs = obs.merge(cell_types, left_on = "cell_subtype", right_index=True)
tdata.obs = obs.loc[tdata.obs_names].copy()

## Drop doublets

In [ ]:
tdata = tdata[tdata.obs.cell_subtype.notna()].copy()
for clone, tree in tdata.obst.items():
    print(f"Compressing tree for clone {clone} with {tree.number_of_nodes()} nodes")
    compressed = compress_single_child_internal_nodes(tree)
    print(f"Compressed tree has {compressed.number_of_nodes()} nodes")
    tdata.obst[clone] = compressed
tdata.write_h5td(data_path / "counts.h5td")

## Preprocessing

In [ ]:
print("normalizing")
sc.pp.normalize_total(tdata, target_sum=2e4)
print("log1p")
sc.pp.log1p(tdata)
tdata.write_h5td(data_path / "log1p.h5td")

Highly variable genes

In [ ]:
hvg = pd.read_csv(data_path / "hvg.csv", header=None, index_col=0).index.tolist()
tdata[:, hvg].write_h5td(data_path / "log1p_hvg.h5td")

## Marker genes

In [ ]:
mean_expr = sc.get.aggregate(tdata, by=["cell_subtype"], func="mean")
mean_expr =pd.DataFrame(mean_expr.layers["mean"].T, index=mean_expr.var_names, columns=mean_expr.obs_names)
logfc_mat = mean_expr.sub(mean_expr.mean(axis=1), axis=0)
logfc = (
    logfc_mat
    .reset_index()
    .melt(id_vars="gene_name", var_name="cell_subtype", value_name="logfc")
    .merge(
        mean_expr.reset_index().melt(
            id_vars="gene_name",
            var_name="cell_subtype",
            value_name="mean"
        ),
        on=["gene_name", "cell_subtype"]
    )
)

# 2) Pearson correlation between subtypes based on logFC profiles
subtype_corr = logfc_mat.corr(method="pearson")

# 3) Find top 3 most similar subtypes for each subtype
top3_similar = {}
for subtype in subtype_corr.columns:
    top3 = (
        subtype_corr[subtype]
        .drop(index=subtype)          # remove self-correlation
        .sort_values(ascending=False)
        .head(3)
        .index
        .tolist()
    )
    top3_similar[subtype] = top3

# 4) Local logFC relative to the 3 most similar cell types
local_logfc_mat = pd.DataFrame(index=mean_expr.index, columns=mean_expr.columns, dtype=float)

for subtype in mean_expr.columns:
    neighbors = top3_similar[subtype]
    ref_expr = mean_expr[neighbors].mean(axis=1)
    local_logfc_mat[subtype] = mean_expr[subtype] - ref_expr

local_logfc = (
    local_logfc_mat
    .reset_index()
    .melt(id_vars="gene_name", var_name="cell_subtype", value_name="logfc")
    .merge(
        mean_expr.reset_index().melt(
            id_vars="gene_name",
            var_name="cell_subtype",
            value_name="mean"
        ),
        on=["gene_name", "cell_subtype"]
    )
)

In [27]:
for df, name in [(logfc, "global"), (local_logfc, "local")]:
    top_markers = (
        df.query("mean > .5")
        .sort_values(["cell_subtype", "logfc"], ascending=[True, False])
        .groupby("cell_subtype")
        .head(10)
        .groupby("cell_subtype")["gene_name"]
        .apply(lambda x: ", ".join(x))
        .reset_index(name=f"{name}_markers")
    )
    cell_types.drop(columns=[f"{name}_markers"], inplace=True, errors="ignore")
    cell_types = cell_types.merge(top_markers, on="cell_subtype")

stage_representation = (
    tdata.obs.groupby("cell_subtype")["stage"]
      .agg(lambda x: ", ".join(pd.unique(x.astype(str))))
)
cell_types["stages"] = cell_types["cell_subtype"].map(stage_representation)
cell_types["n_cells"] = cell_types["cell_subtype"].map(tdata.obs["cell_subtype"].value_counts())
cell_types[["subtype_id","cell_subtype","cell_type","cluster","lineage","germ_layer","stages",
    "n_cells","global_markers","local_markers"]].to_csv(results_path / "cell_type_info.csv")